### 请求模型：客户端可以提交什么
位于 HTTP 输入边界，负责：
- 校验客户端输入
- 将 JSON 转成 Python/Pydantic 对象
- 限制客户端可以提交的字段

In [ ]:
from pydantic import BaseModel, Field


class UserCreateDTO(BaseModel):
    username: str = Field(min_length=2, max_length=30)
    password: str = Field(min_length=8)

### 响应模型：客户端可以看到什么
响应模型会限制和过滤输出数据，响应模型负责：
- 规定接口输出结构
- 过滤内部字段
- 防止密码、成本价等敏感信息泄漏
- 生成 OpenAPI 文档
- 检查服务端返回值是否符合约定

In [ ]:
class UserResponseDTO(BaseModel):
    id: int
    username: str

In [ ]:
@app.post(
    "/users",
    response_model=UserResponseDTO,
    status_code=201,
)
async def create_user(payload: UserCreateDTO):
    ...

### 领域模型：业务中的“用户”是什么
不负责接收 HTTP JSON，也不负责数据库表映射。它负责表达业务状态和业务行为。
领域模型可以使用：
- 普通 Python 类
- dataclass
- Pydantic 模型

In [ ]:
from dataclasses import dataclass


@dataclass
class User:
    id: int
    username: str
    password_hash: str
    is_active: bool = True

    def deactivate(self) -> None:
        if not self.is_active:
            raise ValueError("User is already inactive")

        self.is_active = False

### ORM 模型：数据如何存入数据库
ORM 模型表达的是“数据库如何存储用户”，领域模型表达的是“业务如何理解用户”。它负责：
- 表名
- 列
- 主键和外键
- 索引
- 数据库类型
- 表之间的关系


In [ ]:
class UserORM(Base):
    __tablename__ = "users"

    id = mapped_column(primary_key=True)
    username = mapped_column(unique=True)
    password_hash = mapped_column()
    is_active = mapped_column(default=True)

### 哪些校验放在哪里
#### Pydantic 请求模型
适合不需要访问外部资源的输入格式校验
#### Service
适合业务规则，特别是需要 I/O 的规则
#### Repository
适合需要访问外部资源的规则，负责具体的数据读写
#### ORM 模型
适合数据库表结构和数据校验
#### 响应模型
适合输出格式校验